# ekman spiral

day 30 u(z) plotted against v(z) through the column at each site. the classic ekman signature is a spiral rotating with depth. this is the cleanest proof the coriolis + eddy viscosity setup is doing real momentum physics.

In [ ]:
using JLD2
using Plots

gr()
default(size=(800, 800), linewidth=2)

const ROOT = normpath(joinpath(@__DIR__, ".."))
const SITES = ["lat30lon-50", "lat-25lon-10", "lat-45lon80", "lat0lon-140", "lat30lon-150", "lat40lon-25"]

load_v2(site) = JLD2.load(joinpath(ROOT, "output/era5/ml_forced_30day_v2_qt_$(site).jld2"))

In [ ]:
function ekman_plot(site)
    d = load_v2(site)
    u = reverse(d["u_profiles"][end])
    v = reverse(d["v_profiles"][end])
    vert_res = length(u)
    z = collect(range(0, stop=100, length=vert_res))

    plt = plot(u, v, color=:red, lw=2, marker=:circle, markersize=3,
               xlabel="u (m/s)", ylabel="v (m/s)",
               title="$site — Ekman spiral day 30", titlefontsize=10, legend=false, aspect_ratio=:equal)
    # annotate surface and deepest point
    annotate!(plt, u[1], v[1], text("  surface", 8, :left, :bottom))
    annotate!(plt, u[end], v[end], text("  100m", 8, :left, :top))
    # origin lines
    vline!(plt, [0], color=:gray, linestyle=:dot, alpha=0.5)
    hline!(plt, [0], color=:gray, linestyle=:dot, alpha=0.5)
    plt
end

plot([ekman_plot(s) for s in SITES]..., layout=(3,2), size=(1200,1500))

## ekman vs GLORYS side by side at one site

In [ ]:
function ekman_compare(site)
    sim = load_v2(site)
    gly_f = site == "lat30lon-50" ?
        joinpath(ROOT, "data/generated/glorys_processed.jld2") :
        joinpath(ROOT, "data/generated/glorys_processed_$(site).jld2")
    gly = JLD2.load(gly_f)
    u_s = reverse(sim["u_profiles"][end]); v_s = reverse(sim["v_profiles"][end])
    u_g = gly["u_profiles"][30]; v_g = gly["v_profiles"][30]
    plt = plot(u_s, v_s, color=:red, lw=2, marker=:circle, markersize=3, label="sim v2",
               xlabel="u (m/s)", ylabel="v (m/s)",
               title="$site — sim vs GLORYS spiral", aspect_ratio=:equal, legend=:topright)
    plot!(plt, u_g, v_g, color=:green, lw=2, marker=:square, markersize=3, linestyle=:dash, label="GLORYS")
    vline!(plt, [0], color=:gray, linestyle=:dot, alpha=0.5, label=nothing)
    hline!(plt, [0], color=:gray, linestyle=:dot, alpha=0.5, label=nothing)
    plt
end

ekman_compare("lat30lon-50")